<a href="https://colab.research.google.com/github/ghizlane89/0__GenIA/blob/Bootcamp/W7_D4_DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Étape 1 : importations et chemins de base

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from transformers import BertTokenizer, BertForSequenceClassification, BertConfig
from transformers.models.bert.modeling_bert import BertEncoder
from sklearn.metrics import roc_auc_score

# Détection du device (GPU si disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# Etape 2 ==> iomport du dataset

from google.colab import drive
drive.mount('/content/drive')

# ✅ Chemins vers Google Drive
TRAIN_PATH = "/content/drive/MyDrive/Datasets/train_essays.csv"
TEST_PATH = "/content/drive/MyDrive/Datasets/test_essays.csv"
PROMPT_PATH = "/content/drive/MyDrive/Datasets/train_prompts.csv"

# ✅ Lecture des fichiers
import pandas as pd
src_train = pd.read_csv(TRAIN_PATH)
src_prompt = pd.read_csv(PROMPT_PATH)
src_sub = pd.read_csv(TEST_PATH)


Mounted at /content/drive


In [4]:
# Display basic statistics and structure of the dataset.

# Aperçu du train dataset
print("📊 Train Dataset:")
print("Shape :", src_train.shape)
print("Colonnes :", src_train.columns.tolist())
print("\nPremières lignes :")
print(src_train.head())

# Statistiques de base (longueur texte, label)
print("\nStatistiques descriptives :")
print(src_train.describe(include='all'))

# Distribution des labels (si colonne 'label' présente)
if 'label' in src_train.columns:
    print("\nDistribution des labels :")
    print(src_train['label'].value_counts())




📊 Train Dataset:
Shape : (1378, 4)
Colonnes : ['id', 'prompt_id', 'text', 'generated']

Premières lignes :
         id  prompt_id                                               text  \
0  0059830c          0  Cars. Cars have been around since they became ...   
1  005db917          0  Transportation is a large necessity in most co...   
2  008f63e3          0  "America's love affair with it's vehicles seem...   
3  00940276          0  How often do you ride in a car? Do you drive a...   
4  00c39458          0  Cars are a wonderful thing. They are perhaps o...   

   generated  
0          0  
1          0  
2          0  
3          0  
4          0  

Statistiques descriptives :
              id    prompt_id  \
count       1378  1378.000000   
unique      1378          NaN   
top     ffe1ca0d          NaN   
freq           1          NaN   
mean         NaN     0.486212   
std          NaN     0.499991   
min          NaN     0.000000   
25%          NaN     0.000000   
50%          N

In [5]:
# Aperçu du prompt dataset

print("\n🧪 Test Dataset:")
print("Shape :", src_prompt.shape)
print("Colonnes :", src_prompt.columns.tolist())
print("\nPremières lignes :")
print(src_prompt.head())


🧪 Test Dataset:
Shape : (2, 4)
Colonnes : ['prompt_id', 'prompt_name', 'instructions', 'source_text']

Premières lignes :
   prompt_id                       prompt_name  \
0          0                   Car-free cities   
1          1  Does the electoral college work?   

                                        instructions  \
0  Write an explanatory essay to inform fellow ci...   
1  Write a letter to your state senator in which ...   

                                         source_text  
0  # In German Suburb, Life Goes On Without Cars ...  
1  # What Is the Electoral College? by the Office...  


In [6]:
# Aperçu du test dataset

print("\n🧪 Test Dataset:")
print("Shape :", src_sub.shape)
print("Colonnes :", src_sub.columns.tolist())
print("\nPremières lignes :")
print(src_sub.head())


🧪 Test Dataset:
Shape : (3, 3)
Colonnes : ['id', 'prompt_id', 'text']

Premières lignes :
         id  prompt_id          text
0  0000aaaa          2  Aaa bbb ccc.
1  1111bbbb          3  Bbb ccc ddd.
2  2222cccc          4  CCC ddd eee.


In [7]:
# Étape 3 – Prepare the Model

from transformers import BertTokenizer, BertForSequenceClassification

# 1. Charger le tokenizer BERT
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# 2. Charger le modèle BERT pré-entraîné pour classification de séquence
pretrained_model = BertForSequenceClassification.from_pretrained("bert-base-uncased")

# 3. Extraire le modèle d'embedding (sans la couche de classification)
embedding_model = pretrained_model.bert



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
# Etape 4: Set Hyperparameters


# 📦 Batch sizes
train_batch_size = 32
test_batch_size = 32

# 🚀 Learning rate et momentums
lr = 2e-5         # learning rate (vitesse d’apprentissage)
beta1 = 0.5       # paramètre beta1 de l’optimiseur Adam

# 🧠 Vecteur latent (bruit injecté dans le générateur)
nz = 100          # dimension du vecteur aléatoire d'entrée pour le générateur

# 🔁 Entraînement
num_epochs = 3               # nombre d'époques d'entraînement (peut augmenter si besoin)
num_hidden_layers = 6        # nombre de couches dans le BERT encoder (pour le générateur et le discriminateur)
train_ratio = 0.8            # 80% pour entraînement, 20% pour validation


In [9]:
# Etape 5: Préparation des données

# Créer un Dataset personnalisé

import torch
from torch.utils.data import Dataset, DataLoader

class GANDAIGDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]



In [10]:
# Séparer les données en entraînement et test
train_ratio = 0.8
train_size = int(len(src_train) * train_ratio)

train_texts = src_train["text"][:train_size].tolist()
train_labels = src_train["generated"][:train_size].tolist()

test_texts = src_train["text"][train_size:].tolist()
test_labels = src_train["generated"][train_size:].tolist()

# 3. Création des datasets PyTorch
train_dataset = GANDAIGDataset(train_texts, train_labels)
test_dataset = GANDAIGDataset(test_texts, test_labels)

# 4. Création des DataLoaders
train_batch_size = 32
test_batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=test_batch_size)

In [11]:
import torch
import torch.nn as nn
from transformers import BertConfig
from transformers.models.bert.modeling_bert import BertEncoder

# ✅ 1. Définir le nombre de couches
num_hidden_layers = 4  # Tu peux changer selon ton besoin

# ✅ 2. Créer une configuration BERT réduite
config = BertConfig(
    hidden_size=768,
    num_attention_heads=12,
    intermediate_size=3072,
    num_hidden_layers=num_hidden_layers
)

# ✅ 3. Classe Generator
class Generator(nn.Module):
    def __init__(self, input_dim):  # input_dim = nz (ex. : 100)
        super().__init__()

        self.fc = nn.Linear(input_dim, 256 * 128)

        self.conv_net = nn.Sequential(
            nn.ConvTranspose1d(128, 64, kernel_size=4),
            nn.ReLU(),
            nn.ConvTranspose1d(64, 32, kernel_size=4),
            nn.ReLU(),
            nn.ConvTranspose1d(32, 768, kernel_size=4),
            nn.ReLU()
        )

        self.bert_encoder = BertEncoder(config)

    def forward(self, x):
        x = self.fc(x)                         # [B, 256×128]
        x = x.view(-1, 128, 256)               # [B, C=128, L=256]
        x = self.conv_net(x)                   # [B, 768, L']

        # Attention mask bidon (tout à 1)
        attention_mask = torch.ones(x.size(0), x.size(2)).to(x.device)
        extended_attention_mask = attention_mask[:, None, None, :].float()
        extended_attention_mask = (1.0 - extended_attention_mask) * -10000.0

        # Passer dans le BERT encoder
        output = self.bert_encoder(x.permute(0, 2, 1), attention_mask=extended_attention_mask)
        return output  # contient last_hidden_state



In [12]:
import torch
import torch.nn as nn
from transformers import BertModel

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        # 1. BERT pré-entraîné (non gelé par défaut)
        self.bert = BertModel.from_pretrained("bert-base-uncased")

        # 2. Pooling : représentation de [CLS] (1ère token de chaque séquence)
        # BERT renvoie : last_hidden_state, pooler_output, etc.

        # 3. Tête de classification (prédiction vraie/faux)
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()  # pour prédire une probabilité
        )

    def forward(self, input_embeds):
        # On suppose que input_embeds est déjà un batch de séquences BERT [B, L, 768]
        # et non du texte brut → donc on le passe directement

        # 1. On passe dans BERT (pré-calculé pour les "vrais" textes)
        outputs = self.bert(inputs_embeds=input_embeds)
        cls_output = outputs.last_hidden_state[:, 0, :]  # on prend le [CLS]

        # 2. Classification
        return self.classifier(cls_output)



In [13]:
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel

# 1. Paramètres de base
latent_dim = 100
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Modèles
generator = Generator(input_dim=latent_dim).to(device)
discriminator = Discriminator().to(device)

# 3. Fonction de perte
criterion = nn.BCELoss()

# 4. Optimiseurs
optimizer_G = torch.optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

# 5. Tokenizer BERT
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 6. Modèle BERT
bert_model = AutoModel.from_pretrained("bert-base-uncased").to(device)
bert_model.eval()  # pas d'entraînement

# 7. Nombre d’époques
num_epochs = 5

# 8. Boucle d’entraînement GAN
for epoch in range(num_epochs):
    generator.train()
    discriminator.train()
    total_d_loss = 0
    total_g_loss = 0
    all_preds = []
    all_labels = []

    for texts, labels in train_loader:
        # Tokenisation dynamique
        encoding = tokenizer(list(texts), return_tensors="pt", padding=True, truncation=True, max_length=128)
        input_ids = encoding["input_ids"].to(device)
        attention_mask = encoding["attention_mask"].to(device)
        labels = labels.to(device)
        batch_size = input_ids.size(0)

        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)

        # --- 1. Discriminateur ---
        with torch.no_grad():
            real_embeds = bert_model(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state

        noise = torch.randn(batch_size, latent_dim).to(device)
        fake_embeds = generator(noise)

        # Corriger si fake_embeds est un objet complexe
        if isinstance(fake_embeds, dict) or hasattr(fake_embeds, "last_hidden_state"):
            fake_embeds = fake_embeds.last_hidden_state

        outputs_real = discriminator(real_embeds)
        outputs_fake = discriminator(fake_embeds.detach())

        d_loss_real = criterion(outputs_real, real_labels)
        d_loss_fake = criterion(outputs_fake, fake_labels)
        d_loss = d_loss_real + d_loss_fake

        optimizer_D.zero_grad()
        d_loss.backward()
        optimizer_D.step()

        # --- 2. Générateur ---
        outputs_fake_for_g = discriminator(fake_embeds)
        g_loss = criterion(outputs_fake_for_g, real_labels)

        optimizer_G.zero_grad()
        g_loss.backward()
        optimizer_G.step()

        # --- 3. Statistiques ---
        total_d_loss += d_loss.item()
        total_g_loss += g_loss.item()
        all_preds.extend(outputs_fake_for_g.detach().cpu().numpy())
        all_labels.extend(real_labels.cpu().numpy())

    # Score AUC
    auc = roc_auc_score(all_labels, all_preds)
    print(f"Epoch {epoch+1}: D Loss = {total_d_loss:.4f}, G Loss = {total_g_loss:.4f}, AUC = {auc:.4f}")


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Epoch 1: D Loss = 4.5784, G Loss = 143.2637, AUC = nan


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Epoch 2: D Loss = 41.3151, G Loss = 68.2632, AUC = nan


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Epoch 3: D Loss = 49.2150, G Loss = 24.6110, AUC = nan


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Epoch 4: D Loss = 48.6243, G Loss = 24.1852, AUC = nan
Epoch 5: D Loss = 47.8478, G Loss = 27.3614, AUC = nan


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


In [19]:
from sklearn.metrics import roc_auc_score

def safe_roc_auc_score(y_true, y_pred):
    try:
        # Extraire la première colonne si y_true est au format [[1], [0], ...]
        y_true_flat = [int(x[0]) if isinstance(x, (list, tuple)) else int(x) for x in y_true]
        if len(set(y_true_flat)) < 2:
            print("⚠️ AUC non défini : une seule classe présente dans y_true.")
            return None
        auc = roc_auc_score(y_true_flat, y_pred)
        print(f"AUC : {auc:.4f}")
        return auc
    except Exception as e:
        print(f"Erreur lors du calcul de l'AUC : {e}")
        return None

# Exemple d'appel après l'entraînement
safe_roc_auc_score(all_labels, all_preds)



AUC : 0.6502


/tmp/ipython-input-19-3263864956.py:6: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  y_true_flat = [int(x[0]) if isinstance(x, (list, tuple)) else int(x) for x in y_true]


np.float64(0.6501896333754741)

In [20]:
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel

# Fonction AUC sécurisée
def safe_roc_auc_score(y_true, y_pred):
    try:
        # Extraire la première colonne si y_true est au format [[1], [0], ...]
        y_true_flat = [int(x[0]) if isinstance(x, (list, tuple)) or hasattr(x, '__getitem__') else int(x) for x in y_true]
        if len(set(y_true_flat)) < 2:
            print("⚠️ AUC non défini : une seule classe présente dans y_true.")
            return None
        auc = roc_auc_score(y_true_flat, y_pred)
        print(f"AUC : {auc:.4f}")
        return auc
    except Exception as e:
        print(f"Erreur lors du calcul de l'AUC : {e}")
        return None

# Paramètres
latent_dim = 100
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Modèles (définis ailleurs)
generator = Generator(input_dim=latent_dim).to(device)
discriminator = Discriminator().to(device)

# Optimiseurs et perte
criterion = nn.BCELoss()
optimizer_G = torch.optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

# Tokenizer et BERT
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModel.from_pretrained("bert-base-uncased").to(device)
bert_model.eval()

# Entraînement
num_epochs = 5

for epoch in range(num_epochs):
    generator.train()
    discriminator.train()
    total_d_loss = 0
    total_g_loss = 0
    all_preds = []
    all_labels = []

    for texts, labels in train_loader:
        encoding = tokenizer(list(texts), return_tensors="pt", padding=True, truncation=True, max_length=128)
        input_ids = encoding["input_ids"].to(device)
        attention_mask = encoding["attention_mask"].to(device)
        labels = labels.to(device)
        batch_size = input_ids.size(0)

        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)

        # Embeddings réels
        with torch.no_grad():
            real_embeds = bert_model(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state

        # Génération d'embeddings faux
        noise = torch.randn(batch_size, latent_dim).to(device)
        fake_embeds = generator(noise)
        if isinstance(fake_embeds, dict) or hasattr(fake_embeds, "last_hidden_state"):
            fake_embeds = fake_embeds.last_hidden_state

        # Discriminateur
        outputs_real = discriminator(real_embeds)
        outputs_fake = discriminator(fake_embeds.detach())
        d_loss_real = criterion(outputs_real, real_labels)
        d_loss_fake = criterion(outputs_fake, fake_labels)
        d_loss = d_loss_real + d_loss_fake

        optimizer_D.zero_grad()
        d_loss.backward()
        optimizer_D.step()

        # Générateur
        outputs_fake_for_g = discriminator(fake_embeds)
        g_loss = criterion(outputs_fake_for_g, real_labels)

        optimizer_G.zero_grad()
        g_loss.backward()
        optimizer_G.step()

        # Statistiques
        total_d_loss += d_loss.item()
        total_g_loss += g_loss.item()
        all_preds.extend(outputs_fake_for_g.detach().cpu().numpy())
        all_labels.extend(real_labels.cpu().numpy())

    print(f"\n📊 Epoch {epoch+1}")
    print(f"D Loss = {total_d_loss:.4f}, G Loss = {total_g_loss:.4f}")
    safe_roc_auc_score(all_labels, all_preds)



📊 Epoch 1
D Loss = 7.2639, G Loss = 145.6852
⚠️ AUC non défini : une seule classe présente dans y_true.

📊 Epoch 2
D Loss = 41.4050, G Loss = 80.1763
⚠️ AUC non défini : une seule classe présente dans y_true.

📊 Epoch 3
D Loss = 48.5710, G Loss = 24.2904
⚠️ AUC non défini : une seule classe présente dans y_true.

📊 Epoch 4
D Loss = 48.6123, G Loss = 24.3753
⚠️ AUC non défini : une seule classe présente dans y_true.

📊 Epoch 5
D Loss = 48.6267, G Loss = 24.4188
⚠️ AUC non défini : une seule classe présente dans y_true.


In [23]:


    # --- 4. AUC et Sauvegarde du meilleur modèle ---
    auc = safe_roc_auc_score(all_labels, all_preds)
    print(f"Epoch {epoch+1}: D Loss = {total_d_loss:.4f}, G Loss = {total_g_loss:.4f}")

    if auc is not None and auc > best_auc:
        best_auc = auc
        torch.save(discriminator.state_dict(), "best_discriminator.pth")
        print("💾 Meilleur modèle sauvegardé.")




⚠️ AUC non défini : une seule classe présente dans y_true.
Epoch 1: D Loss = 0.0000, G Loss = 0.0000
